In [ ]:
# THIS PROGRAM IS GOOD BUT STILL REQUIRES SOME FIXES:
# 1. Latex formulas are not rendered correctly in the html output.
# 2. The "Core Innovation" section is sometimes too loose and should be more formal (i.e., it should include more formulas).
# 3. The headers are ridicoulously syntethic in their wording; a fluid summary, perhaps with paragraphs, would be better.
# 4. Key figures and tables from a paper must be reported and referenced to support the summary.
# 5. A better LLM could substantially improve the quality of the summaries. Also, the temperature could be increased a bit to improve creativity.

#---


import requests
import fitz  # pymupdf
from bs4 import BeautifulSoup
from openai import OpenAI
import os
import datetime
import re
import json
import markdown
from weasyprint import HTML
from dotenv import load_dotenv

# Load env variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env file.")

client = OpenAI(api_key=OPENAI_API_KEY)
LLM_MODEL = "gpt-4o-mini"
OUTPUT_DIR = "deepdives"
SAFETY_CAP = 80000      # Max characters to send to LLM to avoid overload (4 characters is approx. 1 token)
TEMPERATURE = 0.5       # Creativity level

# ------------- HELPER FUNCTIONS ----------------

def _create_anchor_slug(title):
    """Creates a URL-friendly slug from a title."""
    s = title.lower()
    s = re.sub(r'[^\w\s-]', '', s)
    s = re.sub(r'[\s_-]+', '-', s).strip('-')
    return s

def latex_to_images(markdown_text):
    """
    Scans markdown for LaTeX formulas ($...$ and $$...$$) and replaces them 
    with <img> tags pointing to a rendering API. 
    This is required for WeasyPrint PDF generation (which cannot run MathJax JS).
    """
    # 1. Replace Block Math: $$...$$
    # We use a non-greedy match (.*?) and DOTALL flag to handle newlines
    pattern_block = r'\$\$(.*?)\$\$'
    
    def replace_block(match):
        formula = match.group(1).strip()
        encoded = urllib.parse.quote(formula)
        # Using CodeCogs API for SVG rendering
        return f'<div style="text-align:center; margin: 1em 0;"><img src="https://latex.codecogs.com/svg.latex?{encoded}" alt="{formula}" /></div>'
    
    markdown_text = re.sub(pattern_block, replace_block, markdown_text, flags=re.DOTALL)

    # 2. Replace Inline Math: $...$
    # We avoid matching across lines to prevent accidental huge matches
    pattern_inline = r'\$([^\$\n]+?)\$'
    
    def replace_inline(match):
        formula = match.group(1).strip()
        encoded = urllib.parse.quote(formula)
        return f'<img style="vertical-align:middle; max-height: 1.2em;" src="https://latex.codecogs.com/svg.latex?{encoded}" alt="{formula}" />'

    markdown_text = re.sub(pattern_inline, replace_inline, markdown_text)
    
    return markdown_text

def fetch_paper_text(url):
    """Downloads paper and extracts text using PyMuPDF."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    print(f"📥 Fetching: {url}")
    pdf_url = url
    
    try:
        # Handle ArXiv
        if "arxiv.org/abs" in url:
            pdf_url = url.replace("/abs/", "/pdf/")
            if not pdf_url.endswith(".pdf"): pdf_url += ".pdf"
        # Handle Generic
        elif not url.lower().endswith(".pdf"):
            try:
                response = requests.get(url, headers=headers, timeout=10)
                soup = BeautifulSoup(response.text, "html.parser")
                found_link = soup.find('a', href=True, string=lambda t: t and ("pdf" in t.lower() or "download" in t.lower()))
                if not found_link:
                    found_link = soup.find('a', href=lambda h: h and ".pdf" in h.lower())
                
                if found_link:
                    href = found_link['href']
                    if href.startswith("http"):
                        pdf_url = href
                    elif href.startswith("/"):
                        from urllib.parse import urlparse
                        parsed = urlparse(url)
                        pdf_url = f"{parsed.scheme}://{parsed.netloc}{href}"
                    else:
                        pdf_url = url.rstrip("/") + "/" + href
                    print(f"   -> Detected PDF Link: {pdf_url}")
            except Exception as e:
                print(f"   ⚠️ Could not scrape landing page: {e}. Trying original URL.")

        response = requests.get(pdf_url, headers=headers, timeout=15)
        response.raise_for_status()
        
        with fitz.open(stream=response.content, filetype="pdf") as doc:
            text = ""
            meta_title = doc.metadata.get('title', '')
            if not meta_title or meta_title.strip() == "":
                meta_title = "Unknown Title"
            for page in doc[:30]:
                text += page.get_text()
                
        return text, meta_title

    except Exception as e:
        return f"[ERROR] {e}", "Error"

def generate_deep_dive_json(text, meta_title, original_url):
    """
    Sends text to LLM and requests a JSON response.
    """
    if text.startswith("[ERROR]"):
        return {
            "title": "Error Processing Paper",
            "authors": "Unknown",
            "analysis": f"Could not extract text: {text}"
        }

    # --- THE PROMPT YOU REQUESTED ---
    prompt = f"""
You are a highly specialized expert curator for **“ets4 Monthly (Economic Time Series Forecasting Monthly),”** a newsletter focused exclusively on **practical and impactful forecasting of economic time series.** 
You are also a seasoned **econometrician, forecaster, and data scientist** with deep experience evaluating new modeling techniques, understanding their assumptions and limits, and interpreting empirical evidence.

Your task is to identify the paper's metadata and produce a **technical Deep Dive** into the research paper provided below.

**OUTPUT FORMAT INSTRUCTIONS:**
You must strictly return a **valid JSON object** containing the following keys:
1. `"title"`: The exact title of the paper.
2. `"authors"`: A string listing the authors.
3. `"analysis"`: A Markdown-formatted string containing the Deep Dive report.

**INSTRUCTIONS FOR THE "analysis" CONTENT:**

Before producing the analysis string, apply the following critical lens:

• **Introduce the foundational concepts** necessary to understand the paper’s contribution.
  – Provide a *brief informal explanation* (intuition first).  
  – Provide a *concise formal explanation* (a few key equations only).  
  – These foundations may be broader than the specific innovation, and should orient a researcher not specialized in this sub-field.

• When analyzing the core idea of the paper, **focus on the intuition, design choices, assumptions, and where the method works or fails**, rather than long derivations.  
  – Highlight examples, counterexamples, and scenarios where the method is brittle or where assumptions are unrealistic.

• Provide **two–three sentences on the literature gap** the paper fills and why it matters.

• Read empirical sections *critically*:  
  – Extract findings from tables and figures.  
  – Pay attention to situations where the method *underperforms*, including in Monte Carlo experiments (and whether the MC design is sound).  
  – Emphasize practical implications and caveats for real-world forecasters.

**STRUCTURE FOR THE "analysis" STRING (Strictly follow this):**

1. **Foundational Concepts (Informal + Formal):**  
   The minimum background a non-specialist economist/forecaster needs to follow the paper. Include 1–3 key formulas max.

2. **The Core Innovation:**  
   The specific modeling contribution. Explain the intuition, what it tries to fix, the assumptions under which it works, and importantly **when and why it may fail**.

3. **Methodology:**  
   Model architecture, estimation strategy, feature engineering, experimental setup. Keep it clear and emphasize design decisions over math. Include 1–3 key formulas max.

4. **Position in the Literature:**  
   Briefly state (2–3 sentences) what gap the paper fills.

5. **Empirical Evidence:**  
   Dataset(s), forecasting setup, evaluation metrics, performance vs. baselines.  
   Highlight **both strengths and weak points**, especially where tables/figures reveal weaknesses or instability.

6. **Critical Takeaway (for Practitioners):**  
   One sentence on why this paper matters—or why it doesn’t—for real-world economic time-series forecasting.

---
**TEXT START**
{text[:SAFETY_CAP]} 
**TEXT END**
"""
    try:
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMPERATURE,
            response_format={"type": "json_object"}
        )
        data = json.loads(response.choices[0].message.content)
        return data
    except Exception as e:
        return {
            "title": meta_title,
            "authors": "Unknown",
            "analysis": f"LLM Generation Error: {e}"
        }

# ------------- OUTPUT GENERATION ----------------

def save_outputs(analyzed_papers, output_formats=['md', 'html', 'pdf']):
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    today = datetime.date.today()
    iso_date_str = today.strftime('%Y-%m-%d')
    main_header_date_str = today.strftime('%B %d, %Y')
    front_matter_date_str = today.strftime('%B %Y')
    base_filename = f"ets4_deepdive_monthly_{iso_date_str}"
    
    # --- 1. BUILD BASE MARKDOWN ---
    front_matter = f"""---
title: "ets4 Deep Dive: {front_matter_date_str}"
date: {iso_date_str}
draft: true
toc: false
---
"""
    lines = [front_matter, f"# ets4 Deep Dive: {main_header_date_str}\n", "\n&nbsp;\n"]
    
    if analyzed_papers:
        lines.append("## In this Issue\n")
        for p in analyzed_papers:
            anchor = _create_anchor_slug(p['title'])
            lines.append(f"* [{p['title']}](#{anchor})")
        lines.append("\n---\n")
    
    for p in analyzed_papers:
        anchor = _create_anchor_slug(p['title'])
        lines.append(f"## {p['title']} {{#{anchor}}}") 
        lines.append(f"[Link to Source ↗]({p['link']})\n")
        lines.append(f"**Authors:** {p['authors']}\n")
        lines.append(p['analysis'])
        lines.append("\n&nbsp;\n")
        lines.append("\n---\n")

    full_markdown = "\n".join(lines)

    # Save Markdown (Raw)
    if 'md' in output_formats:
        md_path = os.path.join(OUTPUT_DIR, base_filename + ".md")
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(full_markdown)
        print(f"✅ Markdown saved: {md_path}")

    # --- 2. BUILD HTML (With MathJax) ---
    if 'html' in output_formats or 'pdf' in output_formats:
        
        # Convert base MD to HTML
        html_body = markdown.markdown(full_markdown, extensions=['tables', 'fenced_code'])
        
        # HTML Template with MathJax Script
        html_content = f"""
        <html>
        <head>
            <meta charset="utf-8">
            <script>
            MathJax = {{
              tex: {{
                inlineMath: [['$', '$'], ['\\(', '\\)']]
              }}
            }};
            </script>
            <script id="MathJax-script" async
              src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-chtml.js">
            </script>
            <style>
                body {{ font-family: 'Helvetica', 'Arial', sans-serif; line-height: 1.6; max-width: 800px; margin: auto; padding: 2em; color: #333; }}
                h1 {{ color: #2c3e50; border-bottom: 2px solid #eee; }}
                h2 {{ color: #2980b9; margin-top: 2em; }}
                pre {{ background: #f4f4f4; padding: 1em; overflow-x: auto; }}
                code {{ background: #f4f4f4; padding: 2px 5px; }}
                blockquote {{ border-left: 4px solid #ddd; padding-left: 1em; color: #777; }}
                img {{ max-width: 100%; }}
            </style>
        </head>
        <body>
            {html_body}
        </body>
        </html>
        """
        
        if 'html' in output_formats:
            html_path = os.path.join(OUTPUT_DIR, base_filename + ".html")
            with open(html_path, "w", encoding="utf-8") as f:
                f.write(html_content)
            print(f"✅ HTML saved: {html_path}")

        # --- 3. BUILD PDF (With Image Replacement) ---
        if 'pdf' in output_formats:
            # For PDF, we must replace LaTeX with Images BEFORE converting to HTML
            # because WeasyPrint cannot run the MathJax Javascript.
            
            print("   ⚙️  Converting LaTeX to images for PDF generation...")
            md_for_pdf = latex_to_images(full_markdown)
            
            # Convert modified MD to HTML
            html_body_pdf = markdown.markdown(md_for_pdf, extensions=['tables', 'fenced_code'])
            
            # Simpler template for PDF (No MathJax needed, images are baked in)
            html_content_pdf = f"""
            <html>
            <head>
                <meta charset="utf-8">
                <style>
                    @page {{ margin: 2cm; }}
                    body {{ font-family: 'Helvetica', 'Arial', sans-serif; font-size: 12px; line-height: 1.5; color: #333; }}
                    h1 {{ color: #2c3e50; border-bottom: 1px solid #ccc; }}
                    h2 {{ color: #2980b9; margin-top: 20px; page-break-after: avoid; }}
                    pre {{ background: #f0f0f0; padding: 10px; border-radius: 4px; font-size: 10px; }}
                    blockquote {{ border-left: 3px solid #ddd; padding-left: 10px; color: #666; }}
                    img {{ max-width: 100%; }}
                </style>
            </head>
            <body>
                {html_body_pdf}
            </body>
            </html>
            """
            
            pdf_path = os.path.join(OUTPUT_DIR, base_filename + ".pdf")
            try:
                HTML(string=html_content_pdf).write_pdf(pdf_path)
                print(f"✅ PDF saved: {pdf_path}")
            except Exception as e:
                print(f"❌ PDF Generation failed: {e}")

# ------------- MAIN ----------------

def deepdive(links_vector, formats=['md', 'html', 'pdf']):
    print(f"🚀 Starting Deep Dive for {len(links_vector)} papers...")
    analyzed_data = []
    
    for i, link in enumerate(links_vector, 1):
        print(f"\n--- Processing Paper {i}/{len(links_vector)} ---")
        text, meta_title = fetch_paper_text(link)
        
        if len(text) < 500:
            print(f"   ⚠️ Text too short for {link}. Skipping.")
            continue
            
        print(f"   🧠 Analyzing content...")
        result = generate_deep_dive_json(text, meta_title, link)
        
        analyzed_data.append({
            "title": result.get("title", meta_title),
            "authors": result.get("authors", "Unknown"),
            "link": link,
            "analysis": result.get("analysis", "Analysis failed.")
        })

    if analyzed_data:
        print("\n💾 Generating Reports...")
        save_outputs(analyzed_data, output_formats=formats)
    else:
        print("No papers analyzed.")

if __name__ == "__main__":
    # Example Links
    my_links = [
        "https://arxiv.org/pdf/2511.07014",
        "https://www.arxiv.org/abs/2511.07678",
    ]
    deepdive(my_links, formats=['md', 'html', 'pdf'])

🚀 Starting Deep Dive for 2 papers...

--- Processing Paper 1/2 ---
📥 Fetching: https://arxiv.org/pdf/2511.07014
   🧠 Analyzing content...

--- Processing Paper 2/2 ---
📥 Fetching: https://www.arxiv.org/abs/2511.07678
   🧠 Analyzing content...

💾 Generating Reports...
✅ Markdown saved: deepdives/ets4_deepdive_monthly_2025-11-25.md
✅ HTML saved: deepdives/ets4_deepdive_monthly_2025-11-25.html
   ⚙️  Converting LaTeX to images for PDF generation...
✅ PDF saved: deepdives/ets4_deepdive_monthly_2025-11-25.pdf
